<a href="https://colab.research.google.com/github/oleksiikartashovde-glitch/Zero-point/blob/main/all.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
# =====================================================================
# ШАГ 0: СИСТЕМНАЯ УСТАНОВКА БИБЛИОТЕКИ В КОЛАБ
# =====================================================================

import cv2
import numpy as np
import pandas as pd

# =====================================================================
# ШАГ 1: ЗАГРУЗКА ИЗОБРАЖЕНИЙ И БИНАРИЗАЦИЯ
# =====================================================================
# ВАЖНО: На панели файлов Colab должны быть файлы zero.png и symbol.png
img_zero = cv2.imread('zero.png', 0)
img_symbol = cv2.imread('symbol.png', 0)

if img_zero is None or img_symbol is None:
    raise FileNotFoundError("Ошибка: Проверьте наличие файлов 'zero.png' и 'symbol.png' в панели Colab!")

# Переводим в бинарные маски (объект — белый, фон — черный)
_, bin_zero = cv2.threshold(img_zero, 127, 255, cv2.THRESH_BINARY)
_, bin_symbol = cv2.threshold(img_symbol, 127, 255, cv2.THRESH_BINARY)

# =====================================================================
# ШАГ 2: ИЗВЛЕЧЕНИЕ ПЛОСКИХ МОМЕНТОВ ХУ И ИХ ЛОГАРИФМИРОВАНИЕ
# =====================================================================
hu_z = cv2.HuMoments(cv2.moments(bin_zero)).flatten()
hu_x = cv2.HuMoments(cv2.moments(bin_symbol)).flatten()

hu_z_log = -np.sign(hu_z) * np.log10(np.abs(hu_z) + 1e-15)
hu_x_log = -np.sign(hu_x) * np.log10(np.abs(hu_x) + 1e-15)

# =====================================================================
# ШАГ 3: СБОРКА ЕДИНОГО ВЕКТОРНОГО ПАСПОРТА ФОРМЫ (7 ПАРАМЕТРОВ)
# =====================================================================
vector_z = hu_z_log
vector_x = hu_x_log

labels = [f'h{i+1}' for i in range(7)]

descriptions = [
    "h1: Компактность / Общая плотность распределения массы силуэта на плоскости",
    "h2: Степень вытянутости (отношение длинной оси объекта к короткой)",
    "h3: Асимметрия изгибов контура (направленный пространственный эксцентриситет)",
    "h4: Степень ортогональности (наличие прямых углов и Т-образных стыков/перекрестий)",
    "h5: Пропорциональный баланс диагональных элементов (динамика/разлет формы)",
    "h6: Симметрия ориентации (характер закрученности хвостов и дериватов силуэта)",
    "h7: Косая симметрия знака (фундаментальный маркер зеркального отражения)"
]

df_params = pd.DataFrame({
    'Метрика': labels,
    'Геометрическое свойство (Композиционный Гештальт)': descriptions,
    'Точка Ноль (Z)': np.round(vector_z, 5),
    'Символ (X)': np.round(vector_x, 5),
    'Линейная разность (Z-X)': np.round(vector_z - vector_x, 5)
})

print("=" * 125)
print(" 1. СИНТЕЗИРОВАННЫЙ ПАСПОРТ ГЕОМЕТРИЧЕСКИХ ИНВАРИАНТОВ ФОРМЫ (ТОЧКА НОЛЬ vs СИМВОЛ):")
print("=" * 125)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
print(df_params.to_string(index=False))
print("-" * 125 + "\n")

# =====================================================================
# ШАГ 4: ПОШАГОВАЯ ДЕМОНСТРАЦИЯ МАТЕМАТИЧЕСКОГО РАСЧЕТА ДИСТАНЦИИ
# =====================================================================
print(" 2. ПОШАГОВЫЙ МАТЕМАТИЧЕСКИЙ РАСЧЕТ В РАСШИРЕННОМ ПРОСТРАНСТВЕ ПРИЗНАКОВ:")
print("-" * 125)

sum_squared_zero = np.sum(vector_z ** 2)
d_max = np.sqrt(sum_squared_zero)

print(f" а) Вычисляем модуль вектора Точки Ноль (Ориентир d_max для 100% шкалы):")
print(f"    • Сумма квадратов всех {len(labels)} параметров Точки Ноль = {sum_squared_zero:.5f}")
print(f"    • Длина опорного вектора Точки Ноль (d_max)      = {d_max:.5f}\n")

diff = vector_z - vector_x
sum_squared_diff = np.sum(diff ** 2)
d_symbol = np.sqrt(sum_squared_diff)

print(f" б) Вычисляем Евклидову дистанцию между вектором Символа и Точкой Ноль (d_symbol):")
print(f"    • Сумма квадратов линейных разностей параметров (Hu) = {sum_squared_diff:.5f}")
print(f"    • Чистая Евклидова дистанция от Точки Ноль до Символа            = {d_symbol:.5f}\n")

# =====================================================================
# ШАГ 5: ИТОГОВЫЙ СЕМАНТИЧЕСКИЙ ВЕРДИКТ
# =====================================================================
raw_percentage = (d_symbol / d_max) * 100
distance_percentage = min(raw_percentage, 100.0)
match_percentage = 100.0 - distance_percentage

print("=" * 125)
print(" 3. ФИНАЛЬНЫЙ МЕТРИЧЕСКИЙ ОТЧЕТ:")
print("=" * 125)
print(f" • ПСИХОЛОГИЧЕСКАЯ / СЕМАНТИЧЕСКАЯ ДИСТАНЦИЯ (D): {distance_percentage:.2f}%")
print(f" • ИТОГОВОЕ ГЕОМЕТРИЧЕСКОЕ СХОДСТВО ГЕШТАЛЬТОВ:    {match_percentage:.2f}%")
print("-" * 125)

if distance_percentage == 0:
    print(" 🟢 Итог: Полное тождество Символа и Точки Ноль.")
elif distance_percentage < 20:
    print(f" ✅ Малая дистанция ({distance_percentage:.2f}%): Структурный изоморфизм подтвержден. Символ находится в зоне когнитивного притяжения Точки Ноль.")
elif distance_percentage < 60:
    print(f" 🟠 Средняя дистанция ({distance_percentage:.2f}%): Объекты имеют общее частотное ядро, но топологические детали разведены.")
else:
    print(" 🔴 Максимальное удаление: Символ ортогонален Точке Ноль. Геометрическое родство отсутствует.")
print("=" * 125)

 1. СИНТЕЗИРОВАННЫЙ ПАСПОРТ ГЕОМЕТРИЧЕСКИХ ИНВАРИАНТОВ ФОРМЫ (ТОЧКА НОЛЬ vs СИМВОЛ):
Метрика                                  Геометрическое свойство (Композиционный Гештальт)  Точка Ноль (Z)  Символ (X)  Линейная разность (Z-X)
     h1        h1: Компактность / Общая плотность распределения массы силуэта на плоскости         2.78175     2.58598                  0.19577
     h2                 h2: Степень вытянутости (отношение длинной оси объекта к короткой)         5.77069     5.27437                  0.49632
     h3      h3: Асимметрия изгибов контура (направленный пространственный эксцентриситет)         8.83903     7.90164                  0.93738
     h4 h4: Степень ортогональности (наличие прямых углов и Т-образных стыков/перекрестий)         9.09502     8.06166                  1.03336
     h5         h5: Пропорциональный баланс диагональных элементов (динамика/разлет формы)        14.99963    14.96261                  0.03702
     h6      h6: Симметрия ориентации (характер зак

In [7]:
# Install mahotas library
!pip install mahotas

## Zernike Moments Calculation (Отдельный Блок)

In [15]:
import mahotas as mh
import pandas as pd
import numpy as np

zernike_radius = 4 # Use the original fixed radius
# The 'order' parameter defaults to 8 in mahotas.features.zernike_moments when not specified,
# which yields 25 moments for real images (m >= 0, (n-m) even).

print(f"Calculating Zernike Moments for radius: {zernike_radius} (default order=8, yielding 25 moments)")
print("=" * 100)

z_z = mh.features.zernike_moments(bin_zero, radius=zernike_radius)
z_x = mh.features.zernike_moments(bin_symbol, radius=zernike_radius)

num_moments = len(z_z) # This will be 25

zernike_labels = [f'z{i+1}' for i in range(num_moments)]

# Generate descriptions based on (n,m) pairs for order 8 (25 moments)
# The order of moments returned by mahotas is typically (n,m) where m >= 0 and (n-m) is even.
# The sequence is: (0,0), (1,1), (2,0), (2,2), (3,1), (3,3), (4,0), (4,2), (4,4),
# (5,1), (5,3), (5,5), (6,0), (6,2), (6,4), (6,6), (7,1), (7,3), (7,5), (7,7),
# (8,0), (8,2), (8,4), (8,6), (8,8)
zernike_descriptions = [
    "z1 (n=0, m=0): Общая масса внутри единичного круга",
    "z2 (n=1, m=1): Линейный радиальный сдвиг массы от центра масс",
    "z3 (n=2, m=0): Радиальная концентрация (Ядро против Периферии)",
    "z4 (n=2, m=2): Квадрупольная полярная асимметрия (2 луча)",
    "z5 (n=3, m=1): Сложный радиальный изгиб внутренних слоев",
    "z6 (n=3, m=3): Треугольная полярная гармоника (3 луча)",
    "z7 (n=4, m=0): Четвертая радиальная мода, нулевая угловая частота (четырехугольная симметрия)",
    "z8 (n=4, m=2): Четвертая радиальная мода, вторая угловая частота",
    "z9 (n=4, m=4): Четвертая радиальная мода, четвертая угловая частота",
    "z10 (n=5, m=1): Пятая радиальная мода, первая угловая частота",
    "z11 (n=5, m=3): Пятая радиальная мода, третья угловая частота",
    "z12 (n=5, m=5): Пятая радиальная мода, пятая угловая частота",
    "z13 (n=6, m=0): Шестая радиальная мода, нулевая угловая частота (шестиугольная симметрия)",
    "z14 (n=6, m=2): Шестая радиальная мода, вторая угловая частота",
    "z15 (n=6, m=4): Шестая радиальная мода, четвертая угловая частота",
    "z16 (n=6, m=6): Шестая радиальная мода, шестая угловая частота",
    "z17 (n=7, m=1): Седьмая радиальная мода, первая угловая частота",
    "z18 (n=7, m=3): Седьмая радиальная мода, третья угловая частота",
    "z19 (n=7, m=5): Седьмая радиальная мода, пятая угловая частота",
    "z20 (n=7, m=7): Седьмая радиальная мода, седьмая угловая частота",
    "z21 (n=8, m=0): Восьмая радиальная мода, нулевая угловая частота (восьмиугольная симметрия)",
    "z22 (n=8, m=2): Восьмая радиальная мода, вторая угловая частота",
    "z23 (n=8, m=4): Восьмая радиальная мода, четвертая угловая частота",
    "z24 (n=8, m=6): Восьмая радиальная мода, шестая угловая частота",
    "z25 (n=8, m=8): Восьмая радиальная мода, восьмая угловая частота"
]

# Ensure descriptions match the number of moments (should be 25)
current_descriptions = zernike_descriptions[:num_moments]
if len(current_descriptions) < num_moments:
    for i in range(len(current_descriptions), num_moments):
        current_descriptions.append(f"z{i+1}: Дополнительный момент Цернике (n=?, m=?)")

df_zernike = pd.DataFrame({
    'Метрика': zernike_labels,
    'Геометрическое свойство': current_descriptions,
    'Точка Ноль (Z)': np.round(z_z, 5),
    'Символ (X)': np.round(z_x, 5),
    'Линейная разность (Z-X)': np.round(z_z - z_x, 5)
})

print("РАДИАЛЬНЫЕ МОМЕНТЫ ЦЕРНИКЕ (ПОЛЯРНЫЙ БАЗИС):")
print("=" * 100)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
print(df_zernike.to_string(index=False))
print("-" * 100)

# =====================================================================
# ПОШАГОВАЯ ДЕМОНСТРАЦИЯ МАТЕМАТИЧЕСКОГО РАСЧЕТА ДИСТАНЦИИ ДЛЯ ZERNIKE
# =====================================================================
print("\n2. ПОШАГОВЫЙ МАТЕМАТИЧЕСКИЙ РАСЧЕТ В РАСШИРЕННОМ ПРОСТРАНСТВЕ ПРИЗНАКОВ (ZERNIKE):")
print("-" * 125)

sum_squared_zero_zernike = np.sum(z_z ** 2)
d_max_zernike = np.sqrt(sum_squared_zero_zernike)

print(f" а) Вычисляем модуль вектора Точки Ноль (Ориентир d_max для 100% шкалы) для Цернике:")
print(f"    • Сумма квадратов всех {num_moments} параметров Точки Ноль = {sum_squared_zero_zernike:.5f}")
print(f"    • Длина опорного вектора Точки Ноль (d_max)      = {d_max_zernike:.5f}\n")

diff_zernike = z_z - z_x
sum_squared_diff_zernike = np.sum(diff_zernike ** 2)
d_symbol_zernike = np.sqrt(sum_squared_diff_zernike)

print(f" б) Вычисляем Евклидову дистанцию между вектором Символа и Точкой Ноль (d_symbol) для Цернике:")
print(f"    • Сумма квадратов линейных разностей параметров (Zernike) = {sum_squared_diff_zernike:.5f}")
print(f"    • Чистая Евклидова дистанция от Точки Ноль до Символа            = {d_symbol_zernike:.5f}\n")

# =====================================================================
# ИТОГОВЫЙ СЕМАНТИЧЕСКИЙ ВЕРДИКТ ДЛЯ ZERNIKE
# =====================================================================
raw_percentage_zernike = (d_symbol_zernike / d_max_zernike) * 100
distance_percentage_zernike = min(raw_percentage_zernike, 100.0)
match_percentage_zernike = 100.0 - distance_percentage_zernike

print("=" * 125)
print("3. ФИНАЛЬНЫЙ МЕТРИЧЕСКИЙ ОТЧЕТ (ZERNIKE):")
print("=" * 125)
print(f" • ПСИХОЛОГИЧЕСКАЯ / СЕМАНТИЧЕСКАЯ ДИСТАНЦИЯ (D): {distance_percentage_zernike:.2f}%")
print(f" • ИТОГОВОЕ ГЕОМЕТРИЧЕСКОЕ СХОДСТВО ГЕШТАЛЬТОВ:    {match_percentage_zernike:.2f}%")
print("-" * 125)

if distance_percentage_zernike == 0:
    print(" 🟢 Итог: Полное тождество Символа и Точки Ноль (Zernike).")
elif distance_percentage_zernike < 20:
    print(f" ✅ Малая дистанция ({distance_percentage_zernike:.2f}%): Структурный изоморфизм подтвержден. Символ находится в зоне когнитивного притяжения Точки Ноль (Zernike).")
elif distance_percentage_zernike < 60:
    print(f" 🟠 Средняя дистанция ({distance_percentage_zernike:.2f}%): Объекты имеют общее частотное ядро, но топологические детали разведены (Zernike).")
else:
    print(" 🔴 Максимальное удаление: Символ ортогонален Точке Ноль. Геометрическое родство отсутствует (Zernike).")
print("=" * 125)

Calculating Zernike Moments for radius: 4 (default order=8, yielding 25 moments)
РАДИАЛЬНЫЕ МОМЕНТЫ ЦЕРНИКЕ (ПОЛЯРНЫЙ БАЗИС):
Метрика                                                                       Геометрическое свойство  Точка Ноль (Z)  Символ (X)  Линейная разность (Z-X)
     z1                                            z1 (n=0, m=0): Общая масса внутри единичного круга         0.31831     0.31831                 -0.00000
     z2                                 z2 (n=1, m=1): Линейный радиальный сдвиг массы от центра масс         0.02069     0.01432                  0.00637
     z3                                z3 (n=2, m=0): Радиальная концентрация (Ядро против Периферии)         0.00152     0.01409                 -0.01258
     z4                                     z4 (n=2, m=2): Квадрупольная полярная асимметрия (2 луча)         0.03469     0.02632                  0.00838
     z5                                      z5 (n=3, m=1): Сложный радиальный изгиб внутренних сло

## Расчет Расстояния Хаусдорфа (Отдельный Блок)

In [17]:
from scipy.spatial.distance import directed_hausdorff

# Преобразуем бинарные изображения в массивы координат точек контура
# cv2.findContours возвращает list of arrays, каждый массив - это точки одного контура.
# Для Hausdorff distance нам нужен один массив точек.

# Для bin_zero
contours_zero, _ = cv2.findContours(bin_zero, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
if not contours_zero:
    print("Ошибка: Контуры для 'zero.png' не найдены.")
    points_zero = np.array([])
else:
    # Объединяем все контуры в один массив точек
    points_zero = np.vstack(contours_zero).squeeze()

# Для bin_symbol
contours_symbol, _ = cv2.findContours(bin_symbol, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
if not contours_symbol:
    print("Ошибка: Контуры для 'symbol.png' не найдены.")
    points_symbol = np.array([])
else:
    # Объединяем все контуры в один массив точек
    points_symbol = np.vstack(contours_symbol).squeeze()

# Проверяем, что есть точки для обоих изображений перед расчетом
if points_zero.shape[0] > 0 and points_symbol.shape[0] > 0:
    # Расчет направленного расстояния Хаусдорфа
    # d(u, v) - максимальное расстояние от точки в u до ближайшей точки в v
    hausdorff_dist_zero_to_symbol = directed_hausdorff(points_zero, points_symbol)[0]
    hausdorff_dist_symbol_to_zero = directed_hausdorff(points_symbol, points_zero)[0]

    # Расстояние Хаусдорфа - максимум из двух направленных расстояний
    symmetric_hausdorff_distance = max(hausdorff_dist_zero_to_symbol, hausdorff_dist_symbol_to_zero)

    print("=" * 100)
    print("РАСЧЕТ РАССТОЯНИЯ ХАУСДОРФА:")
    print("=" * 100)
    print(f"Направленное расстояние Хаусдорфа от Zero к Symbol: {hausdorff_dist_zero_to_symbol:.3f}")
    print(f"Направленное расстояние Хаусдорфа от Symbol к Zero: {hausdorff_dist_symbol_to_zero:.3f}")
    print(f"Симметричное расстояние Хаусдорфа (максимальное расхождение контуров): {symmetric_hausdorff_distance:.3f}")
    print("-" * 100)

    # =====================================================================
    # ПОШАГОВАЯ ДЕМОНСТРАЦИЯ МАТЕМАТИЧЕСКОГО РАСЧЕТА ДИСТАНЦИИ ДЛЯ ХАУСДОРФА
    # =====================================================================
    print("\n2. ПОШАГОВЫЙ МАТЕМАТИЧЕСКИЙ РАСЧЕТ В РАСШИРЕННОМ ПРОСТРАНСТВЕ ПРИЗНАКОВ (ХАУСДОРФ):")
    print("-" * 125)

    # Determine the maximum possible Hausdorff distance for scaling
    # Using the diagonal of the image as the maximum possible distance
    img_height, img_width = bin_zero.shape # Assuming images are of the same size, use bin_zero dimensions
    d_max_hausdorff = np.sqrt(img_height**2 + img_width**2)

    print(f" а) Вычисляем максимальную возможную дистанцию (Ориентир d_max для 100% шкалы):")
    print(f"    • Максимальная диагональ изображения (d_max)      = {d_max_hausdorff:.3f}\n")

    d_symbol_hausdorff = symmetric_hausdorff_distance

    print(f" б) Евклидова дистанция (симметричное расстояние Хаусдорфа):")
    print(f"    • Чистая Евклидова дистанция (Hausdorff)            = {d_symbol_hausdorff:.3f}\n")

    # =====================================================================
    # ИТОГОВЫЙ СЕМАНТИЧЕСКИЙ ВЕРДИКТ ДЛЯ ХАУСДОРФА
    # =====================================================================
    raw_percentage_hausdorff = (d_symbol_hausdorff / d_max_hausdorff) * 100
    distance_percentage_hausdorff = min(raw_percentage_hausdorff, 100.0)
    match_percentage_hausdorff = 100.0 - distance_percentage_hausdorff

    print("=" * 125)
    print("3. ФИНАЛЬНЫЙ МЕТРИЧЕСКИЙ ОТЧЕТ (ХАУСДОРФ):")
    print("=" * 125)
    print(f" • ПСИХОЛОГИЧЕСКАЯ / СЕМАНТИЧЕСКАЯ ДИСТАНЦИЯ (D): {distance_percentage_hausdorff:.2f}%")
    print(f" • ИТОГОВОЕ ГЕОМЕТРИЧЕСКОЕ СХОДСТВО ГЕШТАЛЬТОВ:    {match_percentage_hausdorff:.2f}%")
    print("-" * 125)

    if distance_percentage_hausdorff == 0:
        print(" 🟢 Итог: Полное тождество Символа и Точки Ноль (Hausdorff).")
    elif distance_percentage_hausdorff < 20:
        print(f" ✅ Малая дистанция ({distance_percentage_hausdorff:.2f}%): Структурный изоморфизм подтвержден. Символ находится в зоне когнитивного притяжения Точки Ноль (Hausdorff).")
    elif distance_percentage_hausdorff < 60:
        print(f" 🟠 Средняя дистанция ({distance_percentage_hausdorff:.2f}%): Объекты имеют общее частотное ядро, но топологические детали разведены (Hausdorff).")
    else:
        print(" 🔴 Максимальное удаление: Символ ортогонален Точке Ноль. Геометрическое родство отсутствует (Hausdorff).")
    print("=" * 125)
else:
    print("Недостаточно данных для расчета расстояния Хаусдорфа: один или оба контура пусты.")

РАСЧЕТ РАССТОЯНИЯ ХАУСДОРФА:
Направленное расстояние Хаусдорфа от Zero к Symbol: 427.163
Направленное расстояние Хаусдорфа от Symbol к Zero: 212.850
Симметричное расстояние Хаусдорфа (максимальное расхождение контуров): 427.163
----------------------------------------------------------------------------------------------------

2. ПОШАГОВЫЙ МАТЕМАТИЧЕСКИЙ РАСЧЕТ В РАСШИРЕННОМ ПРОСТРАНСТВЕ ПРИЗНАКОВ (ХАУСДОРФ):
-----------------------------------------------------------------------------------------------------------------------------
 а) Вычисляем максимальную возможную дистанцию (Ориентир d_max для 100% шкалы):
    • Максимальная диагональ изображения (d_max)      = 3574.449

 б) Евклидова дистанция (симметричное расстояние Хаусдорфа):
    • Чистая Евклидова дистанция (Hausdorff)            = 427.163

3. ФИНАЛЬНЫЙ МЕТРИЧЕСКИЙ ОТЧЕТ (ХАУСДОРФ):
 • ПСИХОЛОГИЧЕСКАЯ / СЕМАНТИЧЕСКАЯ ДИСТАНЦИЯ (D): 11.95%
 • ИТОГОВОЕ ГЕОМЕТРИЧЕСКОЕ СХОДСТВО ГЕШТАЛЬТОВ:    88.05%
----------------------------